# Apache Iceberg: история, снимки и файлы


Исследовать метаданные таблицы, версии, физические файлы, разделы и чтение прежней версии.

Нужны `SparkSessionContext`, каталог `lakehouse` и таблица `lakehouse.raw.pg__matches` хотя бы с одним снимком. Все обращения к служебным таблицам Iceberg выполняются только на чтение.

Ограничения: Когда нужно объяснить, из какой версии и каких файлов получены данные. Снимок может быть удалён политикой хранения; не считайте список объектов S3 заменой метаданным Iceberg.


## Основные понятия
Iceberg связывает логическую таблицу с файлами Parquet и хранит согласованные снимки. Чтение метаданных не изменяет таблицу.


In [ ]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F

## Примеры


**1. Расширенное описание**

Получите схему, поставщика, расположение и свойства таблицы. Эти сведения нужны до анализа снимков и физических файлов.



In [ ]:
with SparkSessionContext('example-iceberg-describe') as spark:
    spark.sql("describe table extended lakehouse.raw.pg__matches").show(100, truncate=False)

**2. История снимков**

Выведите моменты переключения текущей версии. Самая верхняя запись должна соответствовать последнему изменению таблицы.



In [ ]:
with SparkSessionContext('example-iceberg-history') as spark:
    spark.sql("""
        select * 
        from lakehouse.raw.pg__matches.history 
        order by made_current_at desc
    """).show(20, False)

**3. Снимки и операции**

Сопоставьте идентификатор снимка, время фиксации и тип операции. Поле `summary` содержит число добавленных или удалённых файлов и записей.



In [ ]:
with SparkSessionContext('example-iceberg-snapshots') as spark:
    spark.sql("""
        select committed_at, snapshot_id, operation, summary 
        from lakehouse.raw.pg__matches.snapshots 
        order by committed_at desc
    """).show(20, False)

**4. Файлы и разделы**

Проверьте формат, число записей и размер каждого файла. Путь показывает физическое размещение, но состав таблицы определяет снимок Iceberg.



In [ ]:
with SparkSessionContext('example-iceberg-files') as spark:
    spark.sql("""
        select content, file_format, record_count, file_size_in_bytes, file_path 
        from lakehouse.raw.pg__matches.files 
        order by file_size_in_bytes desc
    """).show(20, truncate=False)


**5. Чтение снимка**

Возьмите идентификатор последнего снимка и прочитайте именно эту версию. Первые пять строк должны совпадать с обычным чтением текущей таблицы.



In [ ]:
with SparkSessionContext('example-iceberg-time-travel') as spark:
    latest = spark.sql("""
        select max(snapshot_id) as snapshot_id 
        from lakehouse.raw.pg__matches.snapshots
    """).first()["snapshot_id"]

    print("Последний снимок:", latest)
    spark.read.option("snapshot-id", latest).table("lakehouse.raw.pg__matches").limit(5).show(truncate=False)
